In [ ]:
import os
import time
import psutil
import pandas as pd

from openai import OpenAI
from google import genai

# -----------------------------
# API Clients
# -----------------------------

openai_client = OpenAI(
    api_key=os.getenv("API_KEY")
)

gemini_client = genai.Client(
    api_key=os.getenv("AQ.Ab8RN6I-d07i3tA6LInDm_FIPy2KW4eMBhmuHjXXrP4o8P0q9w")
)

PROMPTS = [
    "Explain phishing attacks",
    "Write python code for bubble sort",
    "Explain transformers in AI"
]

# -----------------------------
# Benchmark Function
# -----------------------------

def benchmark_gpt3(prompt):

    process = psutil.Process()

    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = openai_client.completions.create(
        model="davinci-002",
        prompt=prompt,
        max_tokens=150
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.choices[0].text

    return {
        "model":"GPT-3",
        "latency_sec":round(end-start,3),
        "memory_mb":round(
            mem_after-mem_before,
            2
        ),
        "output_chars":len(output),
        "response":output[:150]
    }


def benchmark_gemini(prompt):

    process = psutil.Process()

    mem_before = process.memory_info().rss / 1024**2

    start = time.time()

    response = gemini_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )

    end = time.time()

    mem_after = process.memory_info().rss / 1024**2

    output = response.text

    return {
        "model":"Gemini-2",
        "latency_sec":round(end-start,3),
        "memory_mb":round(
            mem_after-mem_before,
            2
        ),
        "output_chars":len(output),
        "response":output[:150]
    }

# -----------------------------
# Execute Tests
# -----------------------------

results = []

for p in PROMPTS:

    print("\nTesting:", p)

    results.append(
        benchmark_gpt3(p)
    )

    results.append(
        benchmark_gemini(p)
    )

df = pd.DataFrame(results)

print("\n========== RESULTS ==========\n")

print(df)

df.to_csv(
    "benchmark_results.csv",
    index=False
)

print(
    "\nSaved benchmark_results.csv"
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.